In [11]:
"""
Code to query folktables package to fetch ACS data

Code is taken from the whyshift package
Written originally by Jiashuo Liu and Tiangyu Wang
"""
import time
import logging
from typing import Tuple, Dict
from collections import namedtuple
import copy
import pickle

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

## Create ACS data

In [2]:
from folktables import ACSDataSource, ACSPublicCoverage
from folktables import BasicProblem

SCHL_vals = ['SCHL', 'schl_at_least_bachelor', 'schl_at_least_high_school_or_ged', 'schl_postgrad']
CIT_vals = ['us', 'pr', 'abroad', 'citizen', 'not']
ESR_vals = ['employed', 'partial_employed', 'unemployed', 'armed', 'partial_armed', 'no']
rac1p_vals = ['white','black','am_ind','alaska','am_alaska','asian','hawaiian','other','two_or_more']

def add_esr_indicators(t):
    for idx, esr in enumerate(ESR_vals):
        t['ESR_%s'%esr] = t.ESR == (idx+1)

def add_cit_indicators(t):
    for idx, cit in enumerate(CIT_vals):
        t['CIT_%s'%cit] = t.CIT == (idx+1)
        
def add_race_indicators(t):
    for idx, race in enumerate(rac1p_vals):
        t['race_%s'%race] = t.RAC1P == (idx+1)
    return t

def add_school_indicators(t):
    t['schl_at_least_bachelor']=t.SCHL >= 21
    t['schl_at_least_high_school_or_ged']=t.SCHL >= 17
    t['schl_postgrad']=t.SCHL >= 22
    return t

def add_married_indicator(t):
    t['married']=t.MAR==1
    t['widowed']=t.MAR==2
    t['divorced']=t.MAR==3
    t['separated']=t.MAR==4
    t['never']=t.MAR==5
    return t

def add_indicators_pubcov(t):
    add_race_indicators(t)
    add_school_indicators(t)
    add_married_indicator(t)
    add_cit_indicators(t)
    add_esr_indicators(t)
    return t

def public_coverage_filter(data):
    """
    Filters for the public health insurance prediction task; focus on low income Americans, and those not eligible for Medicare
    """
    df = data
    df = df[df['AGEP'] < 65]
    df = df[df['PINCP'] <= 30000]
    return df

def prepare_acs_pubcov(state, year=2018):
    task = ACSPublicCoverage
    data_source = ACSDataSource(survey_year='2018', horizon='1-Year', survey='person')
    source_data = data_source.get_data(states=[state], download=True)
    
    source_data = public_coverage_filter(source_data)
    rac1p_vals = ['white','black','am_ind','alaska','am_alaska','asian','hawaiian','other','two_or_more']
    
    feature_names = ['SEX', 'AGEP', 'DIS', 'ESP', 'MIG', 'MIL', 'ANC', 'NATIVITY', 'DEAR', 'DEYE',
                    'DREM', 'PINCP', 'FER',  'married', 'widowed','divorced','separated','never']+['race_'+x for x in rac1p_vals]+SCHL_vals+['CIT_'+x for x in CIT_vals]\
                    +['ESR_'+x for x in ESR_vals]
    
    new_task = BasicProblem(feature_names, task._target, task._target_transform,
                task._group, task._group_transform,
                preprocess = add_indicators_pubcov, postprocess = task._postprocess)

    X, y, _ = new_task.df_to_numpy(source_data)
    
    Xy = np.concatenate([X,y[:,np.newaxis]],axis=1)
    df = pd.DataFrame(Xy, columns=feature_names+["target"])
    features = [i for i in feature_names+["target"] if (('relp' not in i) and ('occp' not in i) and ('cow' not in i))]
    print(len(features))
    df[features].to_csv("pubcov_%s_%s.csv" % (state, year))
    return df[features]

data_source = ACSDataSource(survey_year='2018', horizon='1-Year', survey='person')
acs_data = data_source.get_data(states=["LA"], download=True)
features, label, group = ACSPublicCoverage.df_to_pandas(acs_data)

In [3]:
features

,AGEP,SCHL,MAR,SEX,DIS,ESP,CIT,MIG,MIL,ANC,NATIVITY,DEAR,DEYE,DREM,PINCP,ESR,ST,FER,RAC1P
0,29.0,18.0,5.0,1.0,2.0,0.0,1.0,1.0,4.0,2.0,1.0,2.0,2.0,2.0,0.0,6.0,22.0,0.0,1.0
1,17.0,13.0,5.0,1.0,2.0,0.0,1.0,1.0,4.0,1.0,1.0,2.0,2.0,2.0,0.0,6.0,22.0,0.0,2.0
2,37.0,13.0,5.0,1.0,2.0,0.0,1.0,1.0,4.0,1.0,1.0,2.0,2.0,2.0,0.0,6.0,22.0,0.0,1.0
3,22.0,19.0,5.0,2.0,2.0,0.0,1.0,1.0,4.0,1.0,1.0,2.0,2.0,2.0,1500.0,6.0,22.0,2.0,1.0
4,33.0,16.0,5.0,1.0,2.0,0.0,1.0,1.0,4.0,1.0,1.0,2.0,2.0,2.0,0.0,6.0,22.0,0.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16874,60.0,13.0,1.0,2.0,2.0,0.0,1.0,1.0,4.0,4.0,1.0,2.0,2.0,2.0,0.0,6.0,22.0,0.0,1.0
16875,41.0,19.0,1.0,2.0,1.0,0.0,1.0,1.0,4.0,4.0,1.0,2.0,2.0,1.0,2500.0,1.0,22.0,2.0,1.0
16876,30.0,16.0,5.0,2.0,2.0,0.0,1.0,1.0,4.0,1.0,1.0,2.0,2.0,2.0,19000.0,6.0,22.0,2.0,2.0
16877,20.0,20.0,1.0,1.0,2.0,0.0,1.0,1.0,4.0,1.0,1.0,2.0,2.0,2.0,15000.0,1.0,22.0,0.0,2.0


In [4]:
sourceXWY_full = prepare_acs_pubcov("NE")
targetXWY_full = prepare_acs_pubcov("LA")

43
43


In [5]:
targetXWY_full = targetXWY_full.drop([
    'race_black',
    'race_am_ind',
    'race_alaska',
    'race_am_alaska',
    'race_asian',
    'race_hawaiian',
    'race_other',
    'race_two_or_more',
], axis=1)
sourceXWY_full = sourceXWY_full.drop([
    'race_black',
    'race_am_ind',
    'race_alaska',
    'race_am_alaska',
    'race_asian',
    'race_hawaiian',
    'race_other',
    'race_two_or_more',
], axis=1)

In [6]:
feature_names_XWY = sourceXWY_full.columns
sourceXWY_full = sourceXWY_full.to_numpy()
targetXWY_full = targetXWY_full.to_numpy()

In [7]:
list(enumerate(feature_names_XWY))

[(0, 'SEX'),
 (1, 'AGEP'),
 (2, 'DIS'),
 (3, 'ESP'),
 (4, 'MIG'),
 (5, 'MIL'),
 (6, 'ANC'),
 (7, 'NATIVITY'),
 (8, 'DEAR'),
 (9, 'DEYE'),
 (10, 'DREM'),
 (11, 'PINCP'),
 (12, 'FER'),
 (13, 'married'),
 (14, 'widowed'),
 (15, 'divorced'),
 (16, 'separated'),
 (17, 'never'),
 (18, 'race_white'),
 (19, 'SCHL'),
 (20, 'schl_at_least_bachelor'),
 (21, 'schl_at_least_high_school_or_ged'),
 (22, 'schl_postgrad'),
 (23, 'CIT_us'),
 (24, 'CIT_pr'),
 (25, 'CIT_abroad'),
 (26, 'CIT_citizen'),
 (27, 'CIT_not'),
 (28, 'ESR_employed'),
 (29, 'ESR_partial_employed'),
 (30, 'ESR_unemployed'),
 (31, 'ESR_armed'),
 (32, 'ESR_partial_armed'),
 (33, 'ESR_no'),
 (34, 'target')]

In [ ]:
feature_df = pd.DataFrame(feature_names_XWY, columns=["vars_name"])
feature_df = feature_df[~feature_df["vars_name"].isin(['SEX','AGEP','race_white'])]
feature_df.insert(loc=0, column="vars", value=np.arange(feature_df.shape[0])+1)
feature_df["vars"] = "X"+feature_df["vars"].astype(str)
feature_df["y_axis_name"] = "Variable"
feature_df.iloc[:-1,:].to_csv("acs_pubcov_feature_names.csv", index=False)

In [9]:
feature_df

,vars,vars_name,y_axis_name
2,X1,DIS,Variable
3,X2,ESP,Variable
4,X3,MIG,Variable
5,X4,MIL,Variable
6,X5,ANC,Variable
7,X6,NATIVITY,Variable
8,X7,DEAR,Variable
9,X8,DEYE,Variable
10,X9,DREM,Variable
11,X10,PINCP,Variable


In [10]:
sourceXWY_full.shape, targetXWY_full.shape

((6332, 35), (16879, 35))

In [12]:
source_val_size = sourceXWY_full.shape[0]//2
target_size = 12000
source_train_size = sourceXWY_full.shape[0]//2

sourceXWY_train, sourceXWY_val = train_test_split(
    sourceXWY_full, train_size=source_train_size, test_size=source_val_size, random_state=0)
targetXWY_modelfix, targetXWY_val = train_test_split(
    targetXWY_full, test_size=target_size, random_state=0)

In [15]:
targetXWY_modelfix.shape, targetXWY_val.shape, sourceXWY_train.shape, sourceXWY_val.shape

((4879, 35), (12000, 35), (3166, 35), (3166, 35))

In [52]:
pd.DataFrame(
    targetXWY_val.astype(float), 
    columns=feature_names_XWY
).to_csv('acs_pubcov_target.csv', index=False)
pd.DataFrame(
    sourceXWY_val.astype(float), 
    columns=feature_names_XWY
).to_csv('acs_pubcov_source_val.csv', index=False)
pd.DataFrame(
    sourceXWY_train.astype(float), 
    columns=feature_names_XWY
).to_csv('acs_pubcov_source_train.csv', index=False)

In [16]:
pd.DataFrame(
    targetXWY_modelfix.astype(float), 
    columns=feature_names_XWY
).to_csv('acs_pubcov_target_modelfix.csv', index=False)

In [54]:
sourceXWY_val, sourceXWY_val.shape

(array([[ 1., 15.,  2., ...,  0.,  0.,  0.],
        [ 2., 45.,  2., ...,  0.,  0.,  0.],
        [ 2., 40.,  2., ...,  0.,  0.,  0.],
        ...,
        [ 1., 48.,  1., ...,  0.,  1.,  1.],
        [ 1., 21.,  2., ...,  0.,  0.,  0.],
        [ 1., 46.,  1., ...,  0.,  0.,  0.]]),
 (3166, 35))

In [55]:
targetXWY_val, targetXWY_val.shape

(array([[ 1., 54.,  2., ...,  0.,  1.,  0.],
        [ 2., 64.,  1., ...,  0.,  1.,  1.],
        [ 2., 17.,  1., ...,  0.,  1.,  0.],
        ...,
        [ 2., 39.,  2., ...,  0.,  0.,  0.],
        [ 2., 24.,  2., ...,  0.,  0.,  1.],
        [ 1., 62.,  2., ...,  0.,  1.,  0.]]),
 (12000, 35))